In [6]:
import os
import torch
import numpy as np
import random

from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Random Seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

# Correct Dataset Path
DATA_DIR = Path("data/W2 Dataset-20261104")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

print("Train Path:", TRAIN_DIR.resolve())
print("Test Path:", TEST_DIR.resolve())

print("Train Exists:", TRAIN_DIR.exists())
print("Test Exists:", TEST_DIR.exists())

# Transform
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load Dataset
train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=transform
)

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))

print("Classes:", train_dataset.classes)

Device: mps
Train Path: /Users/hansamali/Desktop/ai&ml/pusl3123-ai-ml-coursework/AI-ML-3123/assessment1_cifar10/data/W2 Dataset-20261104/train
Test Path: /Users/hansamali/Desktop/ai&ml/pusl3123-ai-ml-coursework/AI-ML-3123/assessment1_cifar10/data/W2 Dataset-20261104/test
Train Exists: False
Test Exists: False


FileNotFoundError: [Errno 2] No such file or directory: 'data/W2 Dataset-20261104/train'

In [4]:
train_size = int(0.8 * len(train_dataset))

val_size = len(train_dataset) - train_size

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_subset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("All DataLoaders Created Successfully!")

NameError: name 'train_dataset' is not defined

In [ ]:
images, labels = next(iter(train_loader))

print("Image Shape:", images.shape)
print("Labels Shape:", labels.shape)

print("Dataset Loading Successful!")

Image Shape: torch.Size([16, 1, 48, 48])
Labels Shape: torch.Size([16])
Dataset Loading Successful!


In [ ]:
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

class CNNClassifier(nn.Module):

    def __init__(self, num_classes=7):

        super(CNNClassifier, self).__init__()

        self.features = nn.Sequential(

            # Convolutional Layer 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Convolutional Layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Convolutional Layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 6 * 6, 256),

            nn.ReLU(),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


model = CNNClassifier(
    num_classes=len(train_dataset.classes)
).to(device)

print(model)

print("CNN Model Created Successfully!")

CNNClassifier(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=7, bias=True)
  )
)
CNN Model Created Successfully!


In [ ]:
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print("Total Trainable Parameters:",
      count_parameters(model))

Total Trainable Parameters: 1274375


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

print("CNN Training Ready!")

CNN Training Ready!


In [ ]:
print("Starting CNN Training...")

for epoch in range(epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)

    train_accuracy = 100 * correct / total

    train_losses.append(train_loss)

    train_accuracies.append(train_accuracy)

    # Validation

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)

    val_accuracy = 100 * val_correct / val_total

    val_losses.append(val_loss)

    val_accuracies.append(val_accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Accuracy: {train_accuracy:.2f}% "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Accuracy: {val_accuracy:.2f}%"
    )

print("CNN Training Completed!")

Starting CNN Training...


OSError: [Errno 89] Operation canceled

In [ ]:
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)

        test_correct += (predicted == labels).sum().item()


test_accuracy = 100 * test_correct / test_total

print("CNN Final Test Accuracy:",
      round(test_accuracy, 2), "%")

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(train_losses, label="Training Loss", marker="o")

plt.plot(val_losses, label="Validation Loss", marker="x")

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.title("CNN Training and Validation Loss")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(train_accuracies, label="Training Accuracy", marker="o")

plt.plot(val_accuracies, label="Validation Accuracy", marker="x")

plt.xlabel("Epochs")

plt.ylabel("Accuracy (%)")

plt.title("CNN Training and Validation Accuracy")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
import os

os.makedirs("results", exist_ok=True)

torch.save(
    model.state_dict(),
    "results/CNN_Emotion_Model.pth"
)

print("CNN Model Saved Successfully!")